# DocFinder — Indexeur Colab (GPU)

Ce notebook tourne sur Colab GPU et prend en charge tout le travail lourd d'embedding.

## Architecture
```
Mac (vieux) ──── /chunks NDJSON ───► Colab GPU ──── embeddings ───► Mac /admin/upsert ───► Qdrant local
              (extraction texte)     (sentence-transformers)         (FastAPI port 8000)    (port 6333)
```

Colab ne se connecte **jamais** directement à Qdrant — tout passe par le serveur FastAPI du Mac.

## Flux automatique
1. Colab poll `/admin/status` toutes les 5s via ngrok
2. Quand `status=running` → fetch `/chunks` en streaming
3. Calcule les embeddings dense sur GPU
4. POST les points vers `/admin/upsert` sur le Mac → le Mac écrit dans Qdrant local

## Prérequis
- Qdrant binaire en cours d'exécution sur le Mac (port 6333)
- Serveur FastAPI en cours d'exécution sur le Mac (port 8000 — `uvicorn server.main:app`)
- **Un seul tunnel ngrok** suffit : DocFinder port 8000
- Collection Qdrant initialisée (`python setup_qdrant.py`)

## 1. Installation des dépendances

In [ ]:
!pip install -q sentence-transformers requests

## 2. Configuration

Remplacez les URLs ngrok par celles affichées sur votre Mac (`ngrok start --all`).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# URL ngrok — à mettre à jour à chaque redémarrage de ngrok sur le Mac
# ─────────────────────────────────────────────────────────────────────────────
DOCFINDER_NGROK_URL = "https://b4be-37-65-61-113.ngrok-free.app"   # port 8000

# Nom de la collection Qdrant
COLLECTION_NAME = "docfinder"

# Modèle d'embedding (identique au serveur local)
MODEL_NAME = "paraphrase-multilingual-mpnet-base-v2"

# Batch GPU : augmenter si la VRAM le permet
BATCH_SIZE = 128

# Accumuler N points avant d'envoyer au Mac (réduit les appels ngrok)
UPSERT_EVERY = 512

# Intervalle de polling (secondes)
POLL_INTERVAL = 5

print(f"DocFinder : {DOCFINDER_NGROK_URL}")

## 3. Chargement du modèle sur GPU

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")

model = SentenceTransformer(MODEL_NAME, device=device)
print(f"Modèle '{MODEL_NAME}' chargé sur {device}")

## 4. Test de connexion au Mac

In [ ]:
import requests

NGROK_HEADERS = {"ngrok-skip-browser-warning": "true"}

r = requests.get(f"{DOCFINDER_NGROK_URL}/health", headers=NGROK_HEADERS, timeout=10)
r.raise_for_status()
data = r.json()
print(f"Serveur DocFinder : {data}")
assert data.get("status") == "ok", "Le serveur ne répond pas correctement"
print("Connexion OK ✓")

## 5. Moteur d'indexation

Consomme le flux NDJSON `/chunks`, calcule les embeddings GPU, écrit dans Qdrant.

In [ ]:
import json
import queue
import threading
import requests

QUEUE_MAXSIZE = 4  # batches en avance max (évite OOM)


def get_status() -> dict:
    """Interroge /admin/status sur le Mac."""
    r = requests.get(
        f"{DOCFINDER_NGROK_URL}/admin/ping",
        headers=NGROK_HEADERS,
        timeout=30,
    )
    return r.json()


def send_heartbeat() -> None:
    """Envoie un heartbeat au Mac avec le device utilisé."""
    try:
        requests.post(
            f"{DOCFINDER_NGROK_URL}/admin/colab/heartbeat",
            json={"device": device},
            headers=NGROK_HEADERS,
            timeout=5,
        )
    except Exception:
        pass  # heartbeat non-bloquant


def check_paused() -> bool:
    """Interroge le Mac pour savoir si une pause a été demandée."""
    try:
        r = requests.get(
            f"{DOCFINDER_NGROK_URL}/admin/colab/control",
            headers=NGROK_HEADERS,
            timeout=5,
        )
        return r.json().get("paused", False)
    except Exception:
        return False


def fetch_indexed_doc_ids() -> set:
    """Récupère les doc_id déjà indexés dans Qdrant (pour reprise)."""
    try:
        r = requests.get(
            f"{DOCFINDER_NGROK_URL}/admin/indexed-doc-ids",
            headers=NGROK_HEADERS,
            timeout=30,
        )
        ids = set(r.json().get("doc_ids", []))
        if ids:
            print(f"  Reprise : {len(ids)} docs déjà indexés — ignorés.")
        return ids
    except Exception as e:
        print(f"  Avertissement : impossible de récupérer les doc_ids existants ({e})")
        return set()


def _stream_producer(path, batch_q: queue.Queue, skip_doc_ids: set) -> None:
    """Thread producteur : stream NDJSON → batches dans la queue."""
    url = f"{DOCFINDER_NGROK_URL}/chunks"
    if path:
        url += f"?path={requests.utils.quote(path)}"
    batch = []
    skipped_docs = 0
    try:
        with requests.get(url, headers=NGROK_HEADERS, stream=True, timeout=300) as resp:
            resp.raise_for_status()
            for raw_line in resp.iter_lines():
                if not raw_line:
                    continue
                line = json.loads(raw_line)
                t = line.get("type")
                if t == "meta":
                    print(f"  {line['total_files']} fichiers à traiter")
                elif t == "file":
                    print(f"  {line['path']} → {line['chunks']} chunks")
                elif t == "skip":
                    print(f"  (ignoré) {line['path']}")
                elif t == "chunk":
                    if line.get("doc_id") in skip_doc_ids:
                        skipped_docs += 1
                        continue  # doc déjà indexé
                    batch.append(line)
                    if len(batch) >= BATCH_SIZE:
                        batch_q.put(batch)
                        batch = []
                elif t == "done":
                    if skipped_docs:
                        print(f"  Flux terminé. ({skipped_docs} chunks ignorés — déjà indexés)")
                    else:
                        print("  Flux terminé.")
                elif t == "error":
                    print(f"  ERREUR flux : {line.get('error')}")
                    break
        if batch:
            batch_q.put(batch)
    except Exception as e:
        print(f"  ERREUR producteur : {e}")
    finally:
        batch_q.put(None)  # sentinel de fin


def _upsert_batch(points_data: list) -> None:
    """Envoie les points au Mac via HTTP — le Mac les écrit dans Qdrant local."""
    r = requests.post(
        f"{DOCFINDER_NGROK_URL}/admin/upsert",
        json=points_data,
        headers=NGROK_HEADERS,
        timeout=120,
    )
    r.raise_for_status()
    resp = r.json()
    if "error" in resp:
        raise RuntimeError(f"Upsert échoué : {resp['error']}")


def index_pipeline(path=None) -> int:
    """
    Pipeline GPU : producteur réseau + consommateur GPU en parallèle.
    - Encode BATCH_SIZE chunks à la fois sur GPU
    - Accumule UPSERT_EVERY points avant d'envoyer au Mac (réduit ngrok round-trips)
    - Libère la mémoire GPU après chaque flush
    - Ignore les docs déjà indexés (reprise sur interruption)
    """
    skip_doc_ids = fetch_indexed_doc_ids()

    batch_q: queue.Queue = queue.Queue(maxsize=QUEUE_MAXSIZE)
    producer = threading.Thread(
        target=_stream_producer, args=(path, batch_q, skip_doc_ids), daemon=True
    )
    producer.start()

    total_inserted = 0
    batch_num = 0
    pending_points: list = []

    while True:
        batch = batch_q.get()
        if batch is None:  # fin du flux
            break

        # ── Pause ──────────────────────────────────────────────────────────
        while check_paused():
            print(f"  ⏸ En pause — en attente de reprise…", end="\r")
            import time as _time
            _time.sleep(3)
        # ───────────────────────────────────────────────────────────────────

        batch_num += 1
        texts = [c["content"] for c in batch]

        embeddings = model.encode(
            texts,
            batch_size=len(texts),
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        for chunk, emb in zip(batch, embeddings):
            p = {
                "id":      chunk["point_id"],
                "dense":   emb.tolist(),
                "payload": {
                    "doc_id":      chunk["doc_id"],
                    "chunk_id":    chunk["chunk_id"],
                    "title":       chunk["title"],
                    "path":        chunk["path"],
                    "abs_path":    chunk.get("abs_path", ""),
                    "doc_type":    chunk["doc_type"],
                    "content":     chunk["content"],
                    "keywords":    chunk["keywords"],
                    "chunk_index": chunk["chunk_index"],
                },
            }
            if chunk.get("sparse_indices"):
                p["sparse_indices"] = chunk["sparse_indices"]
                p["sparse_values"]  = chunk["sparse_values"]
            pending_points.append(p)

        # Libérer la mémoire GPU immédiatement après l'encodage
        del embeddings

        # Flush vers Mac quand on a assez de points
        if len(pending_points) >= UPSERT_EVERY:
            _upsert_batch(pending_points)
            total_inserted += len(pending_points)
            print(f"  Flush {len(pending_points)} points → Qdrant (total : {total_inserted})")
            pending_points = []
            torch.cuda.empty_cache()

    # Flush final du reste
    if pending_points:
        _upsert_batch(pending_points)
        total_inserted += len(pending_points)
        print(f"  Flush final {len(pending_points)} points → Qdrant (total : {total_inserted})")
        pending_points = []

    torch.cuda.empty_cache()
    producer.join()
    return total_inserted


print("Moteur d'indexation pipeliné prêt.")


## 6. Mode daemon — polling automatique

Cette cellule tourne en boucle. Dès que vous cliquez **Lancer l'indexation** dans l'interface Mac (`http://localhost:8000/admin`), Colab détecte le job et traite automatiquement.

**Arrêt** : interrompre le kernel (bouton ■ ou Ctrl+C).

In [ ]:
import time

print("Daemon démarré — en attente d'un job sur le Mac…")
print(f"Poll toutes les {POLL_INTERVAL}s sur {DOCFINDER_NGROK_URL}/admin/status")
print("Lancez une indexation depuis http://localhost:8000/admin")
print("─" * 60)

job_running = False

while True:
    try:
        # Heartbeat — informe le Mac que Colab est connecté + device utilisé
        send_heartbeat()

        status = get_status()
        s = status.get("status")

        if s == "running" and not job_running:
            job_running = True
            print(f"\n[{time.strftime('%H:%M:%S')}] Job détecté ! Démarrage du pipeline…")
            try:
                n = index_pipeline()
                print(f"\n[{time.strftime('%H:%M:%S')}] Terminé — {n} points insérés dans Qdrant.")
            except Exception as e:
                print(f"[{time.strftime('%H:%M:%S')}] ERREUR : {e}")

        elif s in ("done", "error", "idle") and job_running:
            job_running = False
            print(f"[{time.strftime('%H:%M:%S')}] Job {s}. En attente du prochain…")
            print("─" * 60)

        elif s == "running" and job_running:
            done  = status.get("done", 0)
            total = status.get("total", 0)
            pct   = status.get("progress_pct", 0)
            print(f"  Mac : {done}/{total} fichiers ({pct}%)…", end="\r")

    except requests.exceptions.ConnectionError:
        print(f"[{time.strftime('%H:%M:%S')}] Mac inaccessible, nouvelle tentative…")
    except Exception as e:
        print(f"[{time.strftime('%H:%M:%S')}] Erreur réseau : {e}")

    time.sleep(POLL_INTERVAL)


## 7. Indexation manuelle (optionnel)

Si vous ne voulez pas le daemon, lancez une indexation unique ici.

In [ ]:
# Chemin iCloud sur le Mac (optionnel — utilise le défaut si vide)
CUSTOM_PATH = ""  # ex: "/Users/julien/Library/Mobile Documents/com~apple~CloudDocs"

print("Démarrage du pipeline…")
n = index_pipeline(CUSTOM_PATH or None)
print(f"\nTerminé — {n} points insérés.")


## 8. Vérification — test de l'endpoint upsert

In [ ]:
# Vérifie que le Mac est joignable et que le moteur de recherche est prêt
r = requests.get(f"{DOCFINDER_NGROK_URL}/health", headers=NGROK_HEADERS, timeout=10)
r.raise_for_status()
health = r.json()
print(f"Santé serveur : {health}")

# Vérifie que /admin/status répond
r2 = requests.get(f"{DOCFINDER_NGROK_URL}/admin/status", headers=NGROK_HEADERS, timeout=10)
r2.raise_for_status()
status = r2.json()
print(f"Statut job   : {status['status']} — {status['done']}/{status['total']} fichiers, {status['chunks']} chunks")